# Django, Models and Data Definition

## Introduction to Django Models
---

In this lesson, you'll learn how to **define data models** in Django.

Models are Python classes that represent your database tables - they define the structure of your data.

1. [What is a Model](#What-is-a-Model),
    - [Models as Python Classes](#Models-as-Python-Classes),
    - [Model to Database Mapping](#Model-to-Database-Mapping),
2. [Field Types](#Field-Types),
    - [Text Fields](#Text-Fields),
    - [Numeric Fields](#Numeric-Fields),
    - [Date and Time Fields](#Date-and-Time-Fields),
    - [Relationship Fields](#Relationship-Fields),
3. [Field Options](#Field-Options),
    - [Common Options](#Common-Options),
    - [The choices Option](#The-choices-Option),
4. [Meta Class](#Meta-Class),
    - [Ordering and Naming](#Ordering-and-Naming),
    - [Database Table Name](#Database-Table-Name),
5. [🧠 EXERCISE 🧠, Define a Book Model](#🧠-EXERCISE-🧠,-Define-a-Book-Model).

<br>

## What is a Model

---

### Models as Python Classes

---

A **model** is a Python class that inherits from `django.db.models.Model`.

Think of a model like a **blueprint** for your data:
- Each model maps to a **single database table**
- Each attribute of the model represents a **database column**
- Each instance of the model represents a **row** in the table

<br>

| Python Concept | Database Equivalent |
|----------------|--------------------|
| Model class | Table |
| Class attribute (field) | Column |
| Model instance | Row |
| Instance attribute | Cell value |

In [ ]:
# Example: Basic model definition (for illustration)

from django.db import models

class Book(models.Model):
    """A model representing a book in our bookstore."""
    title = models.CharField(max_length=200)  # VARCHAR(200)
    author = models.CharField(max_length=100)  # VARCHAR(100)
    price = models.DecimalField(max_digits=6, decimal_places=2)  # DECIMAL(6,2)
    published_date = models.DateField()  # DATE

<br>

**Key points:**
- Models live in `models.py` inside your app directory
- You don't define an `id` field - Django adds it automatically as a primary key
- Field types determine both Python and database data types

<br>

### Model to Database Mapping

---

Django's **ORM (Object-Relational Mapping)** translates your Python code to SQL.

You never have to write SQL manually - Django does it for you:

<br>

**Your Python code:**
```python
class Book(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
```

**Generated SQL (PostgreSQL):**
```sql
CREATE TABLE catalog_book (
    id SERIAL PRIMARY KEY,
    title VARCHAR(200) NOT NULL,
    author VARCHAR(100) NOT NULL
);
```

<br>

**Table naming convention:** `<app_name>_<model_name>` in lowercase
- App: `catalog`, Model: `Book` → Table: `catalog_book`

<br>

## Field Types

---

Django provides many **field types** to match different kinds of data. Let's explore the most common ones.

### Text Fields

---

| Field Type | Description | Database Type |
|------------|-------------|---------------|
| `CharField` | Short text with max length | VARCHAR |
| `TextField` | Long text without limit | TEXT |
| `EmailField` | Email with validation | VARCHAR(254) |
| `URLField` | URL with validation | VARCHAR(200) |
| `SlugField` | URL-friendly text | VARCHAR(50) |

In [ ]:
# Example: Text field types

from django.db import models

class Article(models.Model):
    # CharField - REQUIRED: max_length parameter
    title = models.CharField(max_length=200)
    
    # TextField - for long content, no max_length needed
    content = models.TextField()
    
    # SlugField - URL-friendly identifier (letters, numbers, hyphens, underscores)
    slug = models.SlugField(max_length=200, unique=True)
    # Example value: "my-first-article"
    
    # EmailField - validates email format
    author_email = models.EmailField()
    # Example value: "author@example.com"

<br>

**Important:** `CharField` always requires `max_length`. This is a common source of errors!

<br>

### Numeric Fields

---

| Field Type | Description | Database Type |
|------------|-------------|---------------|
| `IntegerField` | Integer (-2B to 2B) | INTEGER |
| `PositiveIntegerField` | 0 to 2B | INTEGER |
| `BigIntegerField` | Larger range | BIGINT |
| `FloatField` | Floating point | DOUBLE PRECISION |
| `DecimalField` | Exact decimals | DECIMAL |

In [ ]:
# Example: Numeric field types

from django.db import models

class Product(models.Model):
    name = models.CharField(max_length=200)
    
    # IntegerField - for whole numbers
    stock_quantity = models.IntegerField(default=0)
    
    # PositiveIntegerField - only positive values
    views_count = models.PositiveIntegerField(default=0)
    
    # DecimalField - REQUIRED: max_digits and decimal_places
    # Perfect for money - avoids floating point errors
    price = models.DecimalField(max_digits=10, decimal_places=2)
    # max_digits=10, decimal_places=2 → up to 99999999.99
    
    # FloatField - for scientific calculations where precision is less critical
    weight_kg = models.FloatField(null=True, blank=True)

<br>

**Best practice:** Always use `DecimalField` for money, never `FloatField`!

```python
# BAD: Floating point errors can occur
price = models.FloatField()

# GOOD: Exact decimal representation
price = models.DecimalField(max_digits=10, decimal_places=2)
```

<br>

### Date and Time Fields

---

| Field Type | Description | Database Type |
|------------|-------------|---------------|
| `DateField` | Date only | DATE |
| `DateTimeField` | Date and time | TIMESTAMP |
| `TimeField` | Time only | TIME |
| `DurationField` | Time duration | INTERVAL |

In [ ]:
# Example: Date and time fields

from django.db import models

class Event(models.Model):
    name = models.CharField(max_length=200)
    
    # DateField - just the date (no time)
    event_date = models.DateField()
    # Example: 2026-03-15
    
    # DateTimeField - date and time together
    start_datetime = models.DateTimeField()
    # Example: 2026-03-15 14:30:00
    
    # auto_now_add - automatically set when object is created
    created_at = models.DateTimeField(auto_now_add=True)
    
    # auto_now - automatically update every time object is saved
    updated_at = models.DateTimeField(auto_now=True)

<br>

**Common pattern:** Most models have `created_at` and `updated_at` fields:

```python
class BaseModel(models.Model):
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)
    
    class Meta:
        abstract = True  # Won't create a table for this model
```

<br>

### Relationship Fields

---

Relationships connect models together, just like foreign keys connect tables in a database.

<br>

| Field Type | Relationship | Example |
|------------|--------------|--------|
| `ForeignKey` | Many-to-One | Many books → One author |
| `OneToOneField` | One-to-One | One user → One profile |
| `ManyToManyField` | Many-to-Many | Books ↔ Categories |

In [ ]:
# Example: ForeignKey - Many-to-One relationship

from django.db import models

class Author(models.Model):
    """An author who writes books."""
    name = models.CharField(max_length=100)
    email = models.EmailField(unique=True)

class Book(models.Model):
    """A book written by an author."""
    title = models.CharField(max_length=200)
    
    # ForeignKey - many books can have one author
    # on_delete=CASCADE: if author is deleted, delete their books too
    author = models.ForeignKey(
        Author,
        on_delete=models.CASCADE,
        related_name='books'  # author.books.all() to get all books
    )

<br>

**on_delete options:**

| Option | Behavior |
|--------|----------|
| `CASCADE` | Delete related objects (most common) |
| `PROTECT` | Prevent deletion if related objects exist |
| `SET_NULL` | Set to NULL (requires `null=True`) |
| `SET_DEFAULT` | Set to default value |
| `DO_NOTHING` | Do nothing (may cause errors) |

In [ ]:
# Example: ManyToManyField

from django.db import models

class Category(models.Model):
    """A category for organizing books."""
    name = models.CharField(max_length=100)

class Book(models.Model):
    """A book that can belong to multiple categories."""
    title = models.CharField(max_length=200)
    
    # ManyToManyField - books can have many categories, categories can have many books
    # Django automatically creates an intermediate table
    categories = models.ManyToManyField(
        Category,
        related_name='books'
    )

# Usage:
# book.categories.all()     - get all categories for a book
# category.books.all()      - get all books in a category
# book.categories.add(cat)  - add category to book

<br>

## Field Options

---

### Common Options

---

Every field type accepts certain **options** (keyword arguments) that configure its behavior.

<br>

| Option | Default | Description |
|--------|---------|-------------|
| `null` | `False` | If `True`, allows NULL in database |
| `blank` | `False` | If `True`, allows empty string in forms |
| `default` | - | Default value for the field |
| `unique` | `False` | If `True`, value must be unique |
| `db_index` | `False` | If `True`, creates database index |
| `help_text` | - | Help text for forms |

In [ ]:
# Example: Common field options

from django.db import models

class Book(models.Model):
    # Required field - must have a value
    title = models.CharField(max_length=200)
    
    # Optional field - can be NULL in database and empty in forms
    subtitle = models.CharField(max_length=200, null=True, blank=True)
    
    # Field with default value
    is_published = models.BooleanField(default=False)
    
    # Unique field - no two books can have the same ISBN
    isbn = models.CharField(max_length=13, unique=True)
    
    # Indexed field - faster lookups by this field
    publication_year = models.IntegerField(db_index=True)
    
    # Field with help text - displayed in admin and forms
    summary = models.TextField(
        blank=True,
        help_text="A brief description of the book (max 500 words)"
    )

<br>

**Understanding null vs blank:**

| `null` | `blank` | Meaning |
|--------|---------|--------|
| `False` | `False` | Required everywhere |
| `False` | `True` | Can be empty in forms, but stored as empty string |
| `True` | `False` | NULL in DB, but form requires value (rare) |
| `True` | `True` | Fully optional - NULL in DB, optional in forms |

<br>

**Best practice:** For text fields, prefer `blank=True` over `null=True` to avoid having two "empty" states (NULL and empty string).

<br>

### The choices Option

---

The `choices` option limits a field to a predefined set of values - like a dropdown in forms.

In [ ]:
# Example: Using choices for status field

from django.db import models

class Book(models.Model):
    # Define choices as a list of tuples: (stored_value, display_label)
    STATUS_CHOICES = [
        ('draft', 'Draft'),
        ('review', 'Under Review'),
        ('published', 'Published'),
        ('archived', 'Archived'),
    ]
    
    title = models.CharField(max_length=200)
    
    # CharField with limited choices
    status = models.CharField(
        max_length=20,
        choices=STATUS_CHOICES,
        default='draft'
    )

# Usage:
# book.status            → 'draft'
# book.get_status_display()  → 'Draft'

In [ ]:
# Example: Using TextChoices for cleaner code (Django 3.0+)

from django.db import models

class Book(models.Model):
    # TextChoices - cleaner, more Pythonic approach
    class Status(models.TextChoices):
        DRAFT = 'draft', 'Draft'
        REVIEW = 'review', 'Under Review'
        PUBLISHED = 'published', 'Published'
        ARCHIVED = 'archived', 'Archived'
    
    title = models.CharField(max_length=200)
    
    status = models.CharField(
        max_length=20,
        choices=Status.choices,
        default=Status.DRAFT
    )

# Usage:
# book.status = Book.Status.PUBLISHED
# if book.status == Book.Status.DRAFT: ...

<br>

**Also available:** `IntegerChoices` for integer-based choices:

```python
class Priority(models.IntegerChoices):
    LOW = 1, 'Low'
    MEDIUM = 2, 'Medium'
    HIGH = 3, 'High'
```

<br>

## Meta Class

---

The `Meta` inner class provides **metadata** about your model - how it should behave, be named, or be organized.

### Ordering and Naming

---

In [ ]:
# Example: Meta class options

from django.db import models

class Book(models.Model):
    title = models.CharField(max_length=200)
    publication_date = models.DateField()
    price = models.DecimalField(max_digits=6, decimal_places=2)
    
    class Meta:
        # Default ordering for queries
        # "-" prefix means descending
        ordering = ['-publication_date', 'title']
        # Book.objects.all() returns books ordered by date (newest first), then by title
        
        # Human-readable name (singular)
        verbose_name = 'book'
        
        # Human-readable name (plural)
        verbose_name_plural = 'books'

<br>

**Common Meta options:**

| Option | Description |
|--------|-------------|
| `ordering` | Default sort order |
| `verbose_name` | Human name (singular) |
| `verbose_name_plural` | Human name (plural) |
| `db_table` | Custom database table name |
| `unique_together` | Fields that must be unique together |
| `indexes` | Database indexes to create |
| `abstract` | If `True`, don't create table for this model |

<br>

### Database Table Name

---

In [ ]:
# Example: Custom table name and constraints

from django.db import models

class BookReview(models.Model):
    book = models.ForeignKey('Book', on_delete=models.CASCADE)
    user = models.ForeignKey('auth.User', on_delete=models.CASCADE)
    rating = models.IntegerField()
    comment = models.TextField(blank=True)
    created_at = models.DateTimeField(auto_now_add=True)
    
    class Meta:
        # Custom table name (default would be: catalog_bookreview)
        db_table = 'book_reviews'
        
        # A user can only review a book once
        unique_together = ['book', 'user']
        # Alternative (Django 2.2+):
        # constraints = [
        #     models.UniqueConstraint(fields=['book', 'user'], name='unique_review')
        # ]
        
        # Create index for faster queries
        indexes = [
            models.Index(fields=['book', '-created_at'])
        ]

<br>

## Complete Model Example

---

Let's put it all together with a complete, real-world model example:

In [ ]:
# Example: Complete model for a bookstore app
# File: catalog/models.py

from django.db import models
from django.contrib.auth.models import User


class Author(models.Model):
    """An author who writes books."""
    first_name = models.CharField(max_length=100)
    last_name = models.CharField(max_length=100)
    bio = models.TextField(blank=True)
    date_of_birth = models.DateField(null=True, blank=True)
    website = models.URLField(blank=True)
    
    class Meta:
        ordering = ['last_name', 'first_name']
    
    def __str__(self):
        return f"{self.first_name} {self.last_name}"


class Category(models.Model):
    """A category for organizing books."""
    name = models.CharField(max_length=100, unique=True)
    slug = models.SlugField(max_length=100, unique=True)
    description = models.TextField(blank=True)
    
    class Meta:
        verbose_name_plural = 'categories'
        ordering = ['name']
    
    def __str__(self):
        return self.name


class Book(models.Model):
    """A book in our bookstore catalog."""
    
    class Status(models.TextChoices):
        DRAFT = 'draft', 'Draft'
        PUBLISHED = 'published', 'Published'
        OUT_OF_PRINT = 'out_of_print', 'Out of Print'
    
    # Basic information
    title = models.CharField(max_length=200)
    slug = models.SlugField(max_length=200, unique=True)
    isbn = models.CharField('ISBN', max_length=13, unique=True)
    description = models.TextField(blank=True)
    
    # Relationships
    author = models.ForeignKey(
        Author,
        on_delete=models.CASCADE,
        related_name='books'
    )
    categories = models.ManyToManyField(
        Category,
        related_name='books'
    )
    
    # Pricing and inventory
    price = models.DecimalField(max_digits=8, decimal_places=2)
    stock = models.PositiveIntegerField(default=0)
    
    # Status and dates
    status = models.CharField(
        max_length=20,
        choices=Status.choices,
        default=Status.DRAFT
    )
    publication_date = models.DateField(null=True, blank=True)
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)
    
    class Meta:
        ordering = ['-publication_date', 'title']
        indexes = [
            models.Index(fields=['isbn']),
            models.Index(fields=['status', '-publication_date']),
        ]
    
    def __str__(self):
        return self.title
    
    @property
    def is_in_stock(self) -> bool:
        """Check if book is available for purchase."""
        return self.stock > 0

<br>

**Notice:**
- Each model has a `__str__` method for readable representation
- Related fields use `related_name` for reverse lookups
- The `@property` decorator adds computed attributes
- Comments group related fields together

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Model** | Python class inheriting from `models.Model` |
| **Field types** | `CharField`, `TextField`, `IntegerField`, `DecimalField`, `DateField`, `ForeignKey`, etc. |
| **Field options** | `null`, `blank`, `default`, `unique`, `choices` |
| **Meta class** | `ordering`, `verbose_name`, `db_table`, `unique_together` |
| **Best practices** | Use `DecimalField` for money, add `__str__`, use `related_name` |

<br>

### 🧠 EXERCISE 🧠, Define a Book Model

---

Create a model for a library management system:

1. Create `Library` model with:
   - `name` (CharField, max 200 chars)
   - `address` (TextField)
   - `city` (CharField, max 100 chars)
   - `established_year` (IntegerField)

2. Create `Member` model with:
   - `library` (ForeignKey to Library)
   - `first_name`, `last_name` (CharField)
   - `email` (EmailField, unique)
   - `membership_type` (CharField with choices: 'basic', 'premium', 'student')
   - `joined_date` (DateField, auto set on creation)

3. Add appropriate Meta class with:
   - Ordering by `last_name`, then `first_name`
   - `verbose_name` and `verbose_name_plural`

4. Add `__str__` methods to both models

<details>
    <summary>▶️ Solution</summary>
    
```python
# library/models.py

from django.db import models


class Library(models.Model):
    """A library with books and members."""
    name = models.CharField(max_length=200)
    address = models.TextField()
    city = models.CharField(max_length=100)
    established_year = models.IntegerField()
    
    class Meta:
        verbose_name_plural = 'libraries'
        ordering = ['name']
    
    def __str__(self):
        return f"{self.name} ({self.city})"


class Member(models.Model):
    """A library member."""
    
    class MembershipType(models.TextChoices):
        BASIC = 'basic', 'Basic'
        PREMIUM = 'premium', 'Premium'
        STUDENT = 'student', 'Student'
    
    library = models.ForeignKey(
        Library,
        on_delete=models.CASCADE,
        related_name='members'
    )
    first_name = models.CharField(max_length=100)
    last_name = models.CharField(max_length=100)
    email = models.EmailField(unique=True)
    membership_type = models.CharField(
        max_length=20,
        choices=MembershipType.choices,
        default=MembershipType.BASIC
    )
    joined_date = models.DateField(auto_now_add=True)
    
    class Meta:
        verbose_name = 'member'
        verbose_name_plural = 'members'
        ordering = ['last_name', 'first_name']
    
    def __str__(self):
        return f"{self.first_name} {self.last_name}"
```
</details>

<br>

### 🧠 EXERCISE 🧠, Field Types and Options

---

For each scenario, choose the appropriate field type and options:

1. A product **price** that needs to be exact (e.g., $19.99)
2. A user's **biography** that can be very long and is optional
3. An article **status** that can only be 'draft', 'published', or 'archived'
4. A **publication date** that should automatically record when an article is created
5. A **category** field where each product can belong to multiple categories

<details>
    <summary>▶️ Solution</summary>
    
```python
from django.db import models

# 1. Price - use DecimalField for exact money representation
price = models.DecimalField(max_digits=10, decimal_places=2)

# 2. Biography - TextField for long text, blank=True for optional
biography = models.TextField(blank=True)

# 3. Status - CharField with choices
STATUS_CHOICES = [
    ('draft', 'Draft'),
    ('published', 'Published'),
    ('archived', 'Archived'),
]
status = models.CharField(
    max_length=20,
    choices=STATUS_CHOICES,
    default='draft'
)

# 4. Publication date - DateTimeField with auto_now_add
published_at = models.DateTimeField(auto_now_add=True)

# 5. Categories - ManyToManyField
categories = models.ManyToManyField('Category', related_name='products')
```
</details>

---